In [1]:
import mlflow
import pandas as pd
from mlflow.models import infer_signature
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from mlflow.models import make_metric
from mlflow.metrics.base import MetricValue

In [2]:
data = load_diabetes()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Pack test data into one DataFrame (mlflow.evaluate expects this format)
test_data = X_test.copy()
test_data["target"] = y_test

In [3]:
# Let's set an experiment name
mlflow.set_experiment("diabetes_with_evaluate")

2025/10/21 14:55:37 INFO mlflow.tracking.fluent: Experiment with name 'diabetes_with_evaluate' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///Users/vitale/Projects/fun-with-mlflow/notebooks/mlruns/382729294713986286', creation_time=1761051337526, experiment_id='382729294713986286', last_update_time=1761051337526, lifecycle_stage='active', name='diabetes_with_evaluate', tags={}>

In [4]:
# Train and log a model
with mlflow.start_run(run_name="only_training") as run:
    
    model = RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42)
    model.fit(X_train, y_train)

    signature = infer_signature(X_train, model.predict(X_train))
    
    model_info = mlflow.sklearn.log_model(
        sk_model=model,
        name="rf_model",
        signature=signature,
        input_example=X_train.head(1),
        registered_model_name="DiabetesRandomForestModel"
    )

2025/10/21 14:55:44 INFO mlflow.models.model: Found the following environment variables used during model inference: [OPENAI_API_KEY]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.
Successfully registered model 'DiabetesRandomForestModel'.
Created version '1' of model 'DiabetesRandomForestModel'.


In [5]:
with mlflow.start_run(run_name="model_evaluation") as eval_run:
    
    results = mlflow.models.evaluate(
        model=model_info.model_uri,         # can also be "runs:/<run_id>/model"
        data=test_data,
        targets="target",
        model_type="regressor",             # regressor/classifier
        evaluators=["default"],             # default includes regression metrics
    )

    print("Evaluation metrics:")
    print(results.metrics)
    print("Artifacts logged:", results.artifacts.keys())

    print(f"Run ID: {run.info.run_id}")

2025/10/21 14:56:06 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-d0228323486549a181d66f81601cfd58
2025/10/21 14:56:06 INFO mlflow.tracking.fluent: Use `mlflow.set_active_model` to set the active model to a different one if needed.
2025/10/21 14:56:06 INFO mlflow.models.evaluation.default_evaluator: Testing metrics on first row...
2025/10/21 14:56:06 WARNING mlflow.models.evaluation.evaluators.shap: SHAP or matplotlib package is not installed, so model explainability insights will not be logged.


Evaluation metrics:
{'score': 0.45663677692410665, 'example_count': 89, 'mean_absolute_error': 43.53563736129633, 'mean_squared_error': 2878.820810419161, 'root_mean_squared_error': 53.6546438849347, 'sum_on_target': np.float64(12974.0), 'mean_on_target': np.float64(145.77528089887642), 'r2_score': 0.45663677692410665, 'max_error': 146.06765330773814, 'mean_absolute_percentage_error': 0.39510866710894316}
Artifacts logged: dict_keys([])
Run ID: b3d5ff10653140e38ead16d402af43a2


In [6]:
# you can also evaluate datasets
eval_df = pd.DataFrame({
    "predictions": model.predict(X_test),
    "target": y_test
})

with mlflow.start_run(run_name="dataset_evaluation") as data_eval_run:
    
    results = mlflow.models.evaluate(
        data=eval_df,
        predictions="predictions",
        targets="target",
        model_type="regressor",
    )

    print("Dataset Evaluation metrics:")
    print(results.metrics)

2025/10/21 14:56:09 INFO mlflow.models.evaluation.default_evaluator: Testing metrics on first row...
2025/10/21 14:56:09 WARNING mlflow.models.evaluation.evaluators.shap: SHAP or matplotlib package is not installed, so model explainability insights will not be logged.


Dataset Evaluation metrics:
{'example_count': 89, 'mean_absolute_error': 43.53563736129633, 'mean_squared_error': 2878.820810419161, 'root_mean_squared_error': 53.6546438849347, 'sum_on_target': np.float64(12974.0), 'mean_on_target': np.float64(145.77528089887642), 'r2_score': 0.45663677692410665, 'max_error': 146.06765330773814, 'mean_absolute_percentage_error': 0.39510866710894316}


In [7]:
# create a simple diabetes-specific business metric

def diabetes_clinical_metric(predictions, targets):
    """Simple diabetes metric: percentage of predictions within clinically acceptable range."""
    
    # Clinical threshold: predictions within +-25 units are considered acceptable
    clinical_threshold = 25
    within_clinical_range = abs(predictions - targets) <= clinical_threshold
    clinical_accuracy = within_clinical_range.mean()
    
    return MetricValue(aggregate_results={"clinical_accuracy": clinical_accuracy})

business_metric = make_metric(eval_fn=diabetes_clinical_metric, greater_is_better=True, name="diabetes_clinical")

with mlflow.start_run(run_name="custom_metric_evaluation") as custom_eval_run:
    
    results = mlflow.models.evaluate(
        model=model_info.model_uri,
        data=test_data,
        targets="target",
        model_type="regressor",
        extra_metrics=[business_metric],
    )

    print("Custom Metric Evaluation metrics:")
    print(results.metrics)

2025/10/21 14:56:41 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-d0228323486549a181d66f81601cfd58
2025/10/21 14:56:41 INFO mlflow.tracking.fluent: Use `mlflow.set_active_model` to set the active model to a different one if needed.
2025/10/21 14:56:41 INFO mlflow.models.evaluation.default_evaluator: Testing metrics on first row...
2025/10/21 14:56:41 WARNING mlflow.models.evaluation.evaluators.shap: SHAP or matplotlib package is not installed, so model explainability insights will not be logged.


Custom Metric Evaluation metrics:
{'score': 0.45663677692410665, 'example_count': 89, 'mean_absolute_error': 43.53563736129633, 'mean_squared_error': 2878.820810419161, 'root_mean_squared_error': 53.6546438849347, 'sum_on_target': np.float64(12974.0), 'mean_on_target': np.float64(145.77528089887642), 'r2_score': 0.45663677692410665, 'max_error': 146.06765330773814, 'mean_absolute_percentage_error': 0.39510866710894316, 'diabetes_clinical/clinical_accuracy': np.float64(0.3258426966292135)}
